# March 2022: SMEs Survey

Sample Size: 1723 respondents to ~37 questions

In [765]:
import pandas as pd

In [766]:
df = pd.read_csv('../../data/business_survey_msme/FoB Survey microdata (waves 1-12)/16_FoB Survey_2022_March_microdatapackage./16_FoB Survey_2022_March_weightedmicrodata.csv')
codebook = pd.read_csv('../../data/business_survey_msme/FoB Survey microdata (waves 1-12)/16_FoB Survey_2022_March_microdatapackage./16_FoB Survey_2022_March_codebook.csv')

In [767]:
df = df[df['logged_iso2'] == 'DZ']
df = df[df['eligible_flag_survey']==1]
df.shape

(499, 214)

In [769]:
# Normalize apostrophes in codebook for easier matching
# Replace curly apostrophe (U+2019) with straight apostrophe
codebook['question_text'] = codebook['question_text'].str.replace('\u2019', "'", regex=False)

In [770]:
# Categorize questions into groups
demographic = [
    'How old are you?',
    'What is your gender?',
    'Do you have a long-term spouse or partner?',
    'What is your highest level of completed education?',
    'Which of these best describes your main employment situation?'
]

skills = [
    'Over the past 12 months, have you undergone any training to improve your technology (such as internet or computer) skills?',
    'Which of these kinds of professional groups, if any, are you a part of? (Select all that apply)'
]

business_description = [
    'Why did you start or join this business? Please select up to three reasons that are most important to you.',
    'In which economic sector does this business operate?',
    'In which economic sub-sector does this business operate?',
    'How long ago did this business begin operations?',
    'In general, how many employees does this business currently have (excluding yourself)?',
    'Is this business currently both operating and engaged in revenue generating activities?',
    'Which of these best describes the owners of this business?',
    'Which of these best apply to the owners and managers of this business?',
    'What are the most important challenges your business currently faces? (Please select up to two)'
]

covid_govt_support = [
    'What types of government support has this business received since the start of the COVID-19 pandemic? (Select all that apply)',
    'Do you think the COVID-19 crisis is going to change the use of digital technologies permanently for this business?',
    'Which of the following, if any, did your business experience in your supply chain in 2021? (Select all that apply)'
]

income = [
    'What was the value of total revenues (sales) of this establishment in 2021 in ({currency_symbol})? Please enter a single number.',
    'Compared to 2020, how much did your annual sales change in 2021?',
    'Does this business have outside financing? (financing from someone who does not own or work for your company)? (Select all that apply)',
    'What share of online orders in your business\'s sales?',
    'What is the share of online orders in your business\'s sales?'
]

trade = [
    'Is your business engaged in international trade?',
    'Has this business considered selling goods or services in other countries?',
    'What share of this business\'s annual sales are sales made to customers in other countries?',
    'How many countries does your business export to?',
    'Who does your business export to?',
    'What are the most important challenges your business currently faces to selling in other countries? (Please select up to two)'
]

digital_operations = [
    'Did your business order products or services online during the last 12 months?',
    'What share of this business\'s purchases are made online?',
    'What share of online orders by this business were placed within your home country?',
    'Who does your business sell to via online orders?',
    'What share of online orders received by your business were from your home country?',
    'Which of the following describes this business\'s use of digital platforms (online platforms to facilitate interactions with other firms, individuals, or the government)? (Select all that apply)',
    'What are the main challenges this business faces when using or trying to adopt digital platforms for the sale or purchase of goods and services? (Please select up to two)',
    'What are the main challenges this business might face if it tried to adopt digital platforms for the sale or purchase of goods and services? (Please select up to two)'
]

In [771]:
# # Get variable names for each category
def get_variables_for_questions(question_list, codebook):
    """Get variable names corresponding to questions"""
    variables = []
    for question in question_list:
        if pd.notna(question):  # Skip nan values
            var = codebook[codebook['question_text'] == question]['col_name'].unique()
            if len(var) > 0:
                variables.extend(var.tolist())
    return variables


demographic_vars = get_variables_for_questions(demographic, codebook)
skills_vars = get_variables_for_questions(skills, codebook)
business_description_vars = get_variables_for_questions(business_description, codebook)
covid_govt_support_vars = get_variables_for_questions(covid_govt_support, codebook)
income_vars = get_variables_for_questions(income, codebook)
trade_vars = get_variables_for_questions(trade, codebook)
digital_operations_vars = get_variables_for_questions(digital_operations, codebook)

In [772]:
digital_operations_vars = digital_operations_vars + ['dig_cln_2_numeric_something_not_listed_here', 'dig_cln_2_numeric_no_challenges']

In [773]:

%reload_ext autoreload
%autoreload 2

from visuals import *


# Get question texts for the three variables
edu_question = get_question_for_variable('edu_text', codebook)
bus_rol_question = get_question_for_variable('bus_rol_text', codebook)
hhd_has_spo_question = get_question_for_variable('hhd_has_spo_text', codebook)

In [774]:
# Create category_vars dictionary for plot_category function
category_vars = {
    'skills_vars': skills_vars,
    'business_description_vars': business_description_vars,
    'income_vars': income_vars,
    'covid_govt_support_vars': covid_govt_support_vars,
    'digital_operations_vars': digital_operations_vars,
    'trade_vars': trade_vars
}

In [775]:
log_variables = [
 'logged_iso2',
 'logged_country_name',
 'logged_iso3',
 'eligible_flag_survey']

In [776]:
numeric_cols = 0
numeric_col_names = []
text_col_names = []
text_cols = 0
for cname in df.columns:
    if 'numeric' in cname:
        #print(cname, df[cname].unique())
        numeric_cols += 1
        numeric_col_names.append(cname)
    elif 'text' in cname:
        text_cols += 1
        text_col_names.append(cname)

In [777]:
for col_name in numeric_col_names:
    if col_name not in demographic_vars:
        if col_name not in digital_operations_vars + trade_vars + skills_vars + covid_govt_support_vars + business_description_vars + income_vars:
            print(col_name)

In [778]:
# Create indicator columns for single-select (radio) questions
# This makes radio questions consistent with multi-select questions structure

import json
import re

# Get all variable lists
all_survey_vars = digital_operations_vars + trade_vars + skills_vars + covid_govt_support_vars + business_description_vars + income_vars

# Find all text columns (response columns)
text_vars_to_process = [v for v in all_survey_vars if v.endswith('_text') and '_text_' not in v and v in df.columns]

print(f"Processing {len(text_vars_to_process)} text variables to create indicator columns...")

# Collect all new columns in a dictionary to avoid DataFrame fragmentation
new_columns = {}
# Track columns to drop and codebook entries to add
cols_to_drop = []
new_codebook_rows = []

for var in text_vars_to_process:
    base_name = var.replace('_text', '')
    
    # Check if this already has multi-select structure (multiple _text_ columns)
    multi_select_cols = [col for col in df.columns if col.startswith(f"{base_name}_text_") and col != var]
    
    if len(multi_select_cols) > 0:
        # Already has multi-select structure, skip
        continue
    
    # Check if this is a radio question in the codebook
    question_row = codebook[codebook['col_name'] == var]
    if len(question_row) == 0:
        continue
    
    question_type = question_row['question_type'].iloc[0]
    if question_type != 'radio':
        # Only process radio questions
        continue
        
    question_options = question_row['question_options'].iloc[0]
    
    if pd.isna(question_options):
        continue
    
    # Get the question text to use for all indicator columns
    question_text = question_row['question_text'].iloc[0]
    
    # Parse options - format is like: {"1":"dont_know""2":"no""3":"yes"}
    # This is a JSON-like format where values are the actual response text
    try:
        # Fix malformed JSON (missing commas between entries)
        options_str = str(question_options)
        # Add commas between entries like "value1""key2" -> "value1","key2"
        options_str = re.sub(r'""', '","', options_str)
        options_dict = json.loads(options_str)
        
        # Extract the response values (the actual text responses)
        options = list(options_dict.values())
    except:
        # Fallback to pipe-delimited format if JSON parsing fails
        options = str(question_options).split('|')
    
    # Mark the original numeric column for removal to avoid double counting
    original_numeric_col = f"{base_name}_numeric"
    if original_numeric_col in df.columns:
        cols_to_drop.append(original_numeric_col)
    
    # Create indicator column for each option
    for option in options:
        option_clean = option.strip()
        if not option_clean:
            continue
            
        # Create column name: base_numeric_optionname (using numeric to match existing multi-select pattern)
        col_name_safe = option_clean.lower().replace(' ', '_').replace('-', '_').replace('(', '').replace(')', '').replace(',', '').replace("'", '')
        new_col_name = f"{base_name}_numeric_{col_name_safe}"
        
        # Create indicator: 1 if matches, 0 if doesn't, NaN if original was NaN
        # Use np.nan instead of pd.NA to ensure numeric dtype
        import numpy as np
        new_columns[new_col_name] = df[var].apply(lambda x: 1.0 if pd.notna(x) and str(x).strip() == option_clean else (0.0 if pd.notna(x) else np.nan)).astype('float64')
        
        # Add codebook entry for this new column (same question text as original)
        new_codebook_rows.append({
            'col_name': new_col_name,
            'question_text': question_text,
            'question_type': 'numeric',  # It's now an indicator column
            'question_options': pd.NA
        })
    
    print(f"  Created {len(options)} indicator columns for {var}")

# Add all new columns at once using pd.concat to avoid fragmentation
if new_columns:
    df = pd.concat([df, pd.DataFrame(new_columns)], axis=1)
    print(f"\nAdded {len(new_columns)} indicator columns to dataframe")

# Drop original numeric columns to avoid double counting
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped {len(cols_to_drop)} original numeric columns to avoid double counting")

# Add new codebook entries
if new_codebook_rows:
    new_codebook_df = pd.DataFrame(new_codebook_rows)
    codebook = pd.concat([codebook, new_codebook_df], ignore_index=True)
    print(f"Added {len(new_codebook_rows)} entries to codebook")

print("\nIndicator column creation complete!")

Processing 21 text variables to create indicator columns...
  Created 3 indicator columns for ecm_opl_text
  Created 5 indicator columns for ecm_osh_text
  Created 4 indicator columns for ecm_oor_text
  Created 4 indicator columns for ecm_sel_text
  Created 4 indicator columns for ecm_ror_text
  Created 4 indicator columns for imx_eng_text
  Created 6 indicator columns for con_exp_text
  Created 6 indicator columns for exp_share_text
  Created 5 indicator columns for imx_ctr_text
  Created 4 indicator columns for imx_sel_text
  Created 2 indicator columns for bus_trn_text
  Created 3 indicator columns for dig_prm_text
  Created 11 indicator columns for ind_top_text
  Created 9 indicator columns for ind_oth_text
  Created 4 indicator columns for bus_age_text
  Created 8 indicator columns for bus_emp_alt_text
  Created 2 indicator columns for bus_cls_atl2_text
  Created 4 indicator columns for own_str_text
  Created 4 indicator columns for bus_gen_text
  Created 12 indicator columns for 

In [779]:
# Multiply weights by all numeric columns AND the new indicator columns
# First, collect all columns to weight
cols_to_weight = digital_operations_vars + trade_vars + skills_vars + covid_govt_support_vars + business_description_vars + income_vars

# Add all the newly created indicator columns (base_numeric_optionname pattern)
for var in text_vars_to_process:
    base_name = var.replace('_text', '')
    indicator_cols = [col for col in df.columns if col.startswith(f"{base_name}_numeric_") and col != f"{base_name}_numeric"]
    cols_to_weight.extend(indicator_cols)

print(f"Total columns to weight: {len(cols_to_weight)}")

for col_name in cols_to_weight:
    # Skip if column doesn't exist
    if col_name not in df.columns:
        continue
    
    # Skip if all values are NaN
    if df[col_name].isna().all():
        continue
    
    # Only multiply if column is numeric
    # NaN values will remain NaN after multiplication, which is correct
    # (NaN means ineligible/no response, not zero)
    if pd.api.types.is_numeric_dtype(df[col_name]):
        df[col_name] = df['weight_trim_final'] * df[col_name]

Total columns to weight: 298


In [780]:
# Calculate weighted values grouped by demographic variables
numeric_cols_list = digital_operations_vars + trade_vars + skills_vars + covid_govt_support_vars + business_description_vars + income_vars

# Add all the newly created indicator columns (base_numeric_optionname pattern)
for var in text_vars_to_process:
    base_name = var.replace('_text', '')
    indicator_cols = [col for col in df.columns if col.startswith(f"{base_name}_numeric_") and col != f"{base_name}_numeric"]
    numeric_cols_list.extend(indicator_cols)

# Filter to only numeric columns that exist in df
numeric_cols_to_agg = [col for col in numeric_cols_list if col in df.columns and pd.api.types.is_numeric_dtype(df[col])]

print(f"Total columns to aggregate: {len(numeric_cols_to_agg)}")

# Group by demographics and calculate weighted sums
grouped_weighted = df.groupby(demographic_vars).agg({
    **{col: 'sum' for col in numeric_cols_to_agg},
    'weight_trim_final': 'sum'
}).reset_index()

# Calculate weighted means for all columns at once (avoids DataFrame fragmentation)
weight_col = grouped_weighted['weight_trim_final']
weighted_means = {f'{col}_weighted_mean': grouped_weighted[col] / weight_col for col in numeric_cols_to_agg}
grouped_weighted = pd.concat([grouped_weighted, pd.DataFrame(weighted_means)], axis=1)

print(f"Grouped by {len(demographic_vars)} demographic variables")
print(f"Aggregated {len(numeric_cols_to_agg)} numeric columns")
print(f"Result shape: {grouped_weighted.shape}")
print("\nFirst few rows:")
print(grouped_weighted.head())

Total columns to aggregate: 184
Grouped by 10 demographic variables
Aggregated 184 numeric columns
Result shape: (98, 375)

First few rows:
  dem_age_text  dem_age_numeric sex_text  sex_numeric hhd_has_spo_text  \
0     20 to 29             29.0   Female          2.0               No   
1     20 to 29             29.0   Female          2.0               No   
2     20 to 29             29.0   Female          2.0               No   
3     20 to 29             29.0   Female          2.0               No   
4     20 to 29             29.0   Female          2.0               No   

   hhd_has_spo_numeric                                           edu_text  \
0                  1.0  More than secondary, vocational training or ap...   
1                  1.0                                          Secondary   
2                  1.0                              University or college   
3                  1.0                              University or college   
4                  1.0        

In [781]:
# Create separate dataframes for each demographic variable
demographic_dfs = {}

for demo_var in demographic_vars:
    grouped = df.groupby(demo_var).agg({
        **{col: 'sum' for col in numeric_cols_to_agg},
        'weight_trim_final': 'sum'
    }).reset_index()
    
    weight_col = grouped['weight_trim_final']
    weighted_means = {f'{col}_weighted_mean': grouped[col] / weight_col for col in numeric_cols_to_agg}
    grouped = pd.concat([grouped, pd.DataFrame(weighted_means)], axis=1)
    
    demographic_dfs[demo_var] = grouped
    print(f"{demo_var}: {grouped.shape[0]} groups, {grouped.shape[1]} columns")

print(f"\nCreated {len(demographic_dfs)} demographic dataframes")
print(f"Keys: {list(demographic_dfs.keys())}")

dem_age_text: 7 groups, 366 columns
dem_age_numeric: 7 groups, 366 columns
sex_text: 2 groups, 366 columns
sex_numeric: 2 groups, 366 columns
hhd_has_spo_text: 2 groups, 366 columns
hhd_has_spo_numeric: 2 groups, 366 columns
edu_text: 5 groups, 366 columns
edu_numeric: 5 groups, 366 columns
bus_rol_text: 4 groups, 366 columns
bus_rol_numeric: 4 groups, 366 columns

Created 10 demographic dataframes
Keys: ['dem_age_text', 'dem_age_numeric', 'sex_text', 'sex_numeric', 'hhd_has_spo_text', 'hhd_has_spo_numeric', 'edu_text', 'edu_numeric', 'bus_rol_text', 'bus_rol_numeric']


In [782]:
# Create pre-calculated "All Data" aggregation grouped by log_variables
# Similar structure to demographic_dfs but grouped by response identifiers
all_data_aggregated = df.groupby(log_variables).agg({
    **{col: 'sum' for col in numeric_cols_to_agg},
    'weight_trim_final': 'sum'
}).reset_index()

# Calculate weighted means
weight_col = all_data_aggregated['weight_trim_final']
weighted_means = {f'{col}_weighted_mean': all_data_aggregated[col] / weight_col for col in numeric_cols_to_agg}
all_data_aggregated = pd.concat([all_data_aggregated, pd.DataFrame(weighted_means)], axis=1)

print(f"Pre-calculated 'All Data' aggregation:")
print(f"Grouped by {len(log_variables)} log variables")
print(f"Result shape: {all_data_aggregated.shape}")
print(f"Total respondents: {all_data_aggregated['weight_trim_final'].sum():.1f}")

Pre-calculated 'All Data' aggregation:
Grouped by 4 log variables
Result shape: (1, 369)
Total respondents: 497.6


In [828]:
# Helper function to get question text from variable name
def get_question_for_variable(var_name, codebook):
    """Get question text for a variable from codebook"""
    if var_name in codebook['col_name'].values:
        question = codebook[codebook['col_name'] == var_name]['question_text'].iloc[0]
        if pd.notna(question):
            return question
    return var_name

# Test the function
print("Helper function defined")

Helper function defined


In [829]:
import altair as alt
from IPython.display import display

# Enable data transformer to handle larger datasets
alt.data_transformers.enable('default', max_rows=None)

print("Altair imported and configured for large datasets")

Altair imported and configured for large datasets


In [868]:
# Create a combined dataset with all categories for interactive filtering
# Prepare data for all categories
all_categories_data = []

for category in ['skills', 'business_description', 'income', 'covid_govt_support', 'digital_operations', 'trade']:
    # Get variable list
    var_list = category_vars[f'{category}_vars']
    
    # Find unique questions
    unique_questions = {}
    for v in var_list:
        if '_text_' in v:
            base_name = v.split('_text_')[0]
            if base_name not in unique_questions:
                unique_questions[base_name] = []
            unique_questions[base_name].append(v)
        elif '_numeric_' in v and not v.endswith('_numeric'):
            base_name = v.split('_numeric_')[0]
            if base_name not in unique_questions:
                unique_questions[base_name] = []
            unique_questions[base_name].append(v)
        elif v.endswith('_text'):
            base_name = v.replace('_text', '')
            if base_name not in unique_questions:
                unique_questions[base_name] = [v]
    
    # Calculate total weight
    total_weight = all_data_aggregated['weight_trim_final'].sum()
    
    # Process each question
    for base_name, question_cols in unique_questions.items():
        first_col = question_cols[0] if question_cols else None
        if not first_col or first_col not in df.columns:
            continue
        
        question_text = get_question_for_variable(first_col, codebook)
        
        # Find numeric columns
        numeric_cols = [col for col in all_data_aggregated.columns
                       if (col.startswith(f"{base_name}_text_") or col.startswith(f"{base_name}_numeric_") or col == f"{base_name}_numeric")
                       and not col.endswith('_weighted_mean')
                       and pd.api.types.is_numeric_dtype(all_data_aggregated[col])]
        
        indicator_cols = [col for col in numeric_cols if (f"{base_name}_text_" in col or f"{base_name}_numeric_" in col) and not col.endswith('_weighted_mean')]
        
        if indicator_cols:
            # Multi-select/radio questions
            for num_col in indicator_cols:
                weighted_sum = all_data_aggregated[num_col].sum()
                if weighted_sum > 0:
                    if f"{base_name}_text_" in num_col:
                        response = num_col.replace(f"{base_name}_text_", "").replace("_", " ").title()
                    else:
                        response = num_col.replace(f"{base_name}_numeric_", "").replace("_", " ").title()
                    percentage = (weighted_sum / total_weight) * 100
                    all_categories_data.append({
                        'category': category.replace('_', ' ').title(),
                        'question': question_text,
                        'response': response,
                        'value': percentage
                    })
        else:
            # Single-select questions without indicator columns
            numeric_col = f"{base_name}_numeric"
            text_col = f"{base_name}_text"
            if text_col in df.columns and numeric_col in all_data_aggregated.columns:
                # Get unique responses from original data
                for response_text in df[text_col].dropna().unique():
                    # Find matching rows and calculate weighted sum
                    mask = df[text_col] == response_text
                    if mask.any():
                        # Sum weighted values for this response
                        weighted_sum = df.loc[mask, numeric_col].sum()
                        if weighted_sum > 0:
                            percentage = (weighted_sum / total_weight) * 100
                            all_categories_data.append({
                                'category': category.replace('_', ' ').title(),
                                'question': question_text,
                                'response': str(response_text),
                                'value': percentage
                            })

# Create DataFrame
combined_df = pd.DataFrame(all_categories_data)

# Wrap long question text with line breaks
def wrap_text(text, char_limit=70):
    """Add line breaks after ~char_limit characters at word boundaries"""
    words = text.split()
    lines = []
    current_line = []
    current_length = 0
    
    for word in words:
        if current_length + len(word) + 1 > char_limit and current_line:
            lines.append(' '.join(current_line))
            current_line = [word]
            current_length = len(word)
        else:
            current_line.append(word)
            current_length += len(word) + 1
    
    if current_line:
        lines.append(' '.join(current_line))
    
    return '\n'.join(lines)

# Apply text wrapping to combined_df
combined_df['question'] = combined_df['question'].apply(wrap_text)

print(f"Total rows in combined_df: {len(combined_df)}")
print(f"Categories: {combined_df['category'].unique()}")
print(f"Questions per category:")
for cat in combined_df['category'].unique():
    n_questions = combined_df[combined_df['category'] == cat]['question'].nunique()
    print(f"  {cat}: {n_questions} questions")

Total rows in combined_df: 181
Categories: ['Skills' 'Business Description' 'Income' 'Covid Govt Support'
 'Digital Operations' 'Trade']
Questions per category:
  Skills: 2 questions
  Business Description: 9 questions
  Income: 3 questions
  Covid Govt Support: 3 questions
  Digital Operations: 8 questions
  Trade: 6 questions


In [909]:
# Add bucketing for numeric multi-column questions
print("\nAdding bucketed responses for numeric questions...")

def get_bucket_label(value, buckets):
    try:
        num_value = float(value)
        for min_val, max_val, label in buckets:
            if min_val <= num_value <= max_val:
                return label
        return None
    except (ValueError, TypeError):
        return None

# Define buckets for multi-column numeric questions
bucket_configs = {
    'business_profits': {
        'base_col': 'imx_prm',
        'buckets': [
            (0, 0, '0'),
            (1, 10_000, '1-10,000'),
            (10_001, 50_000, '10,001-50,000'),
            (50_001, 100_000, '50,001-100,000'),
            (100_001, 250_000, '100,001-250,000'),
            (250_001, 500_000, '250,001-500,000'),
            (500_001, 1_000_000, '500,001-1,000,000'),
            (1_000_001, float('inf'), '1,000,001+')
        ]
    },
    'business_loans': {
        'base_col': 'imx_loa',
        'buckets': [
            (0, 0, '0'),
            (1, 10_000, '1-10,000'),
            (10_001, 50_000, '10,001-50,000'),
            (50_001, 100_000, '50,001-100,000'),
            (100_001, 250_000, '100,001-250,000'),
            (250_001, 500_000, '250,001-500,000'),
            (500_001, 1_000_000, '500,001-1,000,000'),
            (1_000_001, float('inf'), '1,000,001+')
        ]
    }
}

bucketed_rows = []
total_weight = all_data_aggregated['weight_trim_final'].sum()

for question_key, config in bucket_configs.items():
    base_col = config['base_col']
    buckets = config['buckets']
    
    # Find all numeric columns for this base column
    numeric_cols = [col for col in all_data_aggregated.columns 
                   if col.startswith(f"{base_col}_") and not col.endswith('_weighted_mean')]
    
    if not numeric_cols:
        print(f"  No numeric columns found for {question_key}")
        continue
    
    bucket_aggregation = {}
    
    for col in numeric_cols:
        try:
            # Extract numeric value from column name (e.g., "imx_prm_50000" → 50000)
            value_str = col.replace(f"{base_col}_", "").replace("_", "")
            if value_str == '' or not value_str.replace('.', '').replace('e', '').replace('-', '').isdigit():
                continue
            
            response_value = float(value_str)
            weighted_sum = all_data_aggregated[col].sum()
            
            if weighted_sum > 0:
                bucket_label = get_bucket_label(response_value, buckets)
                if bucket_label:
                    bucket_aggregation[bucket_label] = bucket_aggregation.get(bucket_label, 0) + weighted_sum
        except Exception as e:
            continue
    
    # Find the actual question text
    if 'loans' in question_key:
        question_pattern = 'loans'
    else:
        question_pattern = 'profits'
    
    actual_question = None
    for q in combined_df['question'].unique():
        if question_pattern in q.lower():
            actual_question = q
            break
    
    if actual_question:
        for bucket_label, total_value in sorted(bucket_aggregation.items()):
            percentage = (total_value / total_weight) * 100
            bucketed_rows.append({
                'category': 'Income',
                'question': actual_question,
                'response': bucket_label,
                'value': percentage
            })
        print(f"  ✓ Created {len(bucket_aggregation)} buckets for {question_key}")

# Remove original income question rows that will be replaced
questions_to_remove = set()
for q in combined_df['question'].unique():
    if 'loans' in q.lower() or 'profits' in q.lower():
        questions_to_remove.add(q)

for question in questions_to_remove:
    combined_df = combined_df[combined_df['question'] != question]

# Add bucketed rows
if bucketed_rows:
    bucketed_income_df = pd.DataFrame(bucketed_rows)
    combined_df = pd.concat([combined_df, bucketed_income_df], ignore_index=True)
    print(f"\nAdded {len(bucketed_rows)} bucketed response rows")
else:
    print("\nNo bucketed rows to add")


Adding bucketed responses for numeric questions...
  No numeric columns found for business_profits
  No numeric columns found for business_loans

No bucketed rows to add


In [907]:
import textwrap

# Wrap question text to multiple lines (80 characters per line)
combined_df['question_wrapped'] = combined_df['question'].apply(
    lambda x: textwrap.fill(x, width=80) if x else x
)



In [965]:
# Helper function to create category chart
def create_category_chart(category_name, data, y_max=None):
    """Create a chart for a specific category"""
    cat_data = data[data['category'] == category_name]

    if y_max is None:
        y_max = float(cat_data['value'].max()) if not cat_data.empty else 0
    
    chart = alt.Chart(cat_data).mark_bar().encode(
        x=alt.X('response:N', 
                title=None,
                axis=alt.Axis(labelAngle=-45, labelLimit=200, labelAlign='right')),
        y=alt.Y('value:Q', 
                title='Percentage (%)',
                scale=alt.Scale(domain=[0, y_max])),
        color=alt.Color('response:N', 
                        scale=alt.Scale(scheme='tableau20'),
                        legend=None),
        tooltip=[
            alt.Tooltip('question:N', title='Question'),
            alt.Tooltip('response:N', title='Response'),
            alt.Tooltip('value:Q', title='Percentage', format='.1f')
        ]
    ).properties(
        width=250,
        height=300
    ).facet(
        facet=alt.Facet('question_wrapped:N', header=alt.Header(labelLimit=450, 
                                                                labelFontSize=14, titleOrient='top', labelLineHeight=16)),
        columns=2,
        spacing={'row': 80, 'column': 20}
    ).resolve_scale(
        x='independent',
        y='independent'
    ).properties(
        title={
            "text": f"Algeria SME Survey Results (2022) - {category_name}",
            "fontSize": 16,
            "fontWeight": "bold",
            "anchor": "middle"
        }
    ).configure_view(
        strokeWidth=0
    ).configure_axis(
        labelPadding=10,
        titlePadding=15
    ).configure(
        padding={'left': 10, 'right': 10, 'top': 50, 'bottom': 100}
    )
    
    return chart

print("Helper function created for generating category charts")

Helper function created for generating category charts


## Skills

In [966]:
chart = create_category_chart('Skills', combined_df)
display(chart)

alt.FacetChart(...)

## Business Description

In [967]:
chart = create_category_chart('Business Description', combined_df)
display(chart)

alt.FacetChart(...)

## Income

In [968]:
chart = create_category_chart('Income', combined_df)
display(chart)

alt.FacetChart(...)

## Covid Government Support

In [969]:
chart = create_category_chart('Covid Govt Support', combined_df)
display(chart)

alt.FacetChart(...)

## Digital Operations

In [970]:
chart = create_category_chart('Digital Operations', combined_df)
display(chart)

alt.FacetChart(...)

## Trade

In [971]:
chart = create_category_chart('Trade', combined_df)
display(chart)


alt.FacetChart(...)

In [972]:
text_col_names_wo_dem = []
for x in text_col_names:
    if x not in ['bus_rol_text', 'edu_text', 'dem_age_text', 'hhd_has_spo_text', 'sex_text']:
        text_col_names_wo_dem.append(x)

In [973]:
# Save data for the dashboard
combined_df.to_pickle('combined_df.pkl')
all_data_aggregated.to_pickle('all_data_aggregated.pkl')

print(f"Saved combined_df with {len(combined_df)} rows")
print(f"Saved all_data_aggregated with {all_data_aggregated.shape[0]} rows, {all_data_aggregated.shape[1]} columns")
print(f"\nCategories: {combined_df['category'].unique()}")
print(f"\nQuestions per category:")
for cat in combined_df['category'].unique():
    n_q = combined_df[combined_df['category'] == cat]['question'].nunique()
    print(f"  {cat}: {n_q} questions")

Saved combined_df with 181 rows
Saved all_data_aggregated with 1 rows, 369 columns

Categories: ['Skills' 'Business Description' 'Income' 'Covid Govt Support'
 'Digital Operations' 'Trade']

Questions per category:
  Skills: 2 questions
  Business Description: 9 questions
  Income: 3 questions
  Covid Govt Support: 3 questions
  Digital Operations: 8 questions
  Trade: 6 questions


In [974]:
# Export charts for each category as HTML for the dashboard
import os

# Create output directory
os.makedirs('dashboard_charts', exist_ok=True)

# Save combined_df as CSV for the dashboard
combined_df.to_csv('dashboard_charts/combined_df.csv', index=False)

# Create and save a chart for each category
for category in combined_df['category'].unique():
    print(f"Creating chart for {category}...")
    category_data = combined_df[combined_df['category'] == category]
    
    chart = alt.Chart(category_data).mark_bar().encode(
        x=alt.X('response:N', 
                title=None,
                axis=alt.Axis(labelAngle=-45, labelLimit=150, labelAlign='right')),
        y=alt.Y('value:Q', 
                title='Percentage (%)',
                scale=alt.Scale(domain=[0, 50])),
        color=alt.Color('response:N', 
                        scale=alt.Scale(scheme='tableau20'),
                        legend=None),
        tooltip=[
            alt.Tooltip('question:N', title='Question'),
            alt.Tooltip('response:N', title='Response'),
            alt.Tooltip('value:Q', title='Percentage', format='.1f')
        ]
    ).properties(
        width=400,
        height=300
    ).facet(
        facet=alt.Facet('question:N', header=alt.Header(labelLimit=400, labelFontSize=12, titleOrient='top')),
        columns=2,
        spacing={'row': 80, 'column': 20}
    ).resolve_scale(
        x='independent',
        y='independent'
    ).configure_view(
        strokeWidth=0
    ).configure_axis(
        labelPadding=10,
        titlePadding=15
    ).configure(
        padding={'left': 10, 'right': 10, 'top': 20, 'bottom': 80}
    )
    
    # Save as HTML
    filename = f"dashboard_charts/{category.lower().replace(' ', '_')}.html"
    chart.save(filename)
    print(f"  Saved {filename}")

print("\nAll charts exported successfully!")

Creating chart for Skills...
  Saved dashboard_charts/skills.html
Creating chart for Business Description...
  Saved dashboard_charts/business_description.html
Creating chart for Income...
  Saved dashboard_charts/income.html
Creating chart for Covid Govt Support...
  Saved dashboard_charts/covid_govt_support.html
Creating chart for Digital Operations...
  Saved dashboard_charts/digital_operations.html
Creating chart for Trade...
  Saved dashboard_charts/trade.html

All charts exported successfully!


In [975]:
# Save charts as PNG files for each category
import altair as alt
import os

# Create output directory
os.makedirs('survey_charts', exist_ok=True)

# Save each category's chart as PNG
categories_map = {
    'Skills': 'skills',
    'Trade': 'trade', 
    'Digital Operations': 'digital_operations',
    'Covid Govt Support': 'covid_govt_support',
    'Business Description': 'business_description',
    'Income': 'income'
}

for display_name, category_key in categories_map.items():
    print(f"Creating PNG for {display_name}...")
    
    # Filter data for this category
    category_data = combined_df[combined_df['category'] == display_name]
    
    if len(category_data) > 0:
        # Create chart
        chart = alt.Chart(category_data).mark_bar().encode(
            x=alt.X('response:N', 
                    title=None,
                    axis=alt.Axis(labelAngle=-45, labelLimit=300, labelAlign='right')),
            y=alt.Y('value:Q', 
                    title='Percentage (%)',
                    scale=alt.Scale(domain=[0, 50])),
            color=alt.Color('response:N', 
                            scale=alt.Scale(scheme='tableau20'),
                            legend=None),
            tooltip=[
                alt.Tooltip('question:N', title='Question'),
                alt.Tooltip('response:N', title='Response'),
                alt.Tooltip('value:Q', title='Percentage', format='.1f')
            ]
        ).properties(
            width=450,
            height=350
        ).facet(
            facet=alt.Facet('question:N', 
                           header=alt.Header(labelLimit=600, labelFontSize=11, titleOrient='top')),
            columns=2,
            spacing={'row': 100, 'column': 30}
        ).resolve_scale(
            x='independent',
            y='independent'
        ).configure_view(
            strokeWidth=0
        ).configure_axis(
            labelPadding=10,
            titlePadding=15
        ).configure(
            padding={'left': 10, 'right': 10, 'top': 30, 'bottom': 100}
        )
        
        # Save as PNG
        filename = f"survey_charts/{category_key}.png"
        chart.save(filename, scale_factor=2.0)
        print(f"  Saved {filename}")
    else:
        print(f"  No data available for {display_name}")

print("\nAll charts saved as PNG files in 'survey_charts/' directory!")

Creating PNG for Skills...
  Saved survey_charts/skills.png
Creating PNG for Trade...
  Saved survey_charts/trade.png
Creating PNG for Digital Operations...
  Saved survey_charts/digital_operations.png
Creating PNG for Covid Govt Support...
  Saved survey_charts/covid_govt_support.png
Creating PNG for Business Description...
  Saved survey_charts/business_description.png
Creating PNG for Income...
  Saved survey_charts/income.png

All charts saved as PNG files in 'survey_charts/' directory!
